# Problem Set 5 Solution: Autoencoders
In this problem set solution, we implement and train a deterministic **Autoencoder (AE)** on MNIST handwritten digit images. We build a symmetric encoder–decoder architecture using Flux.jl, train it by minimising mean-squared reconstruction loss, and analyse the learned bottleneck representation through reconstruction quality and latent-space interpolation. The AE establishes the compress-then-reconstruct pattern that underlies the more powerful Variational Autoencoder studied in the paper from our lab.

> __Learning Objectives__
>
> By the end of this problem set, you should be able to:
> * __Implement an encoder and decoder using Flux.jl:__ Write `encode` and `decode` functions that map inputs to a low-dimensional bottleneck and back, using `Chain` and `Dense` layers.
> * __Implement and minimise a reconstruction loss:__ Compute mean-squared error between the input and the reconstruction, and write a Flux.jl training loop that minimises it.
> * __Visualise and interpret the learned latent space:__ Project bottleneck codes onto two principal components and interpolate between pairs of images in latent space to test whether the bottleneck is smooth.

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading the MNIST dataset, and setting up the required constants.

> __Environment Setup with Include.jl__
>
> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of `Include.jl` in the notebook's global scope. `Include.jl` sets paths, loads required external packages, and includes `src/Types.jl` and `src/Compute.jl`, which define the `MyAEModel` type and the helper functions used throughout this notebook.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we use [Flux.jl](https://fluxml.ai/Flux.jl/stable/) for automatic differentiation and neural network layers, [MLDatasets.jl](https://juliaml.github.io/MLDatasets.jl/stable/) to load MNIST, and [Plots.jl](https://docs.juliaplots.org/stable/) for visualization.

### Implementations
The notebook uses the following helper functions from `src/Compute.jl`. Student-implemented functions (`encode`, `decode`, `reconstruction_loss`) are defined directly in notebook cells.

| Function | Source | Description |
|----------|--------|-------------|
| `build_ae_model(input_dim, hidden_dim, latent_dim)` | `src/Compute.jl` | Constructs a `MyAEModel` with a symmetric encoder–decoder architecture. |
| `load_mnist_digit(digit; n_examples)` | `src/Compute.jl` | Loads MNIST training images for one digit class; returns a `(784 × N)` Float32 matrix. |
| `show_image_grid(X; nrows, ncols)` | `src/Compute.jl` | Displays columns of a `784 × N` matrix as a grid of 28×28 greyscale images. |

### Constants
Let's set the constants that control the dataset, model architecture, and training schedule.

In [ ]:
DIGIT         = 3;       # MNIST digit class to model (0–9)
K             = 100;     # number of training examples
D             = 784;     # input dimension: 28 × 28 = 784 pixels
L             = 8;       # bottleneck (latent) dimension
H             = 256;     # hidden-layer width
LR            = 1f-3;    # Adam learning rate
NUM_EPOCHS    = 2_000;   # training epochs

Random.seed!(42);

___
## Background: Autoencoders

An autoencoder is a neural network trained to reproduce its own input at the output layer,
subject to passing through a low-dimensional **bottleneck**. It consists of two sub-networks:

| Component | Maps | Role |
|-----------|------|------|
| **Encoder** $f_\theta$ | $\mathbf{x}\in\mathbb{R}^D \to \mathbf{z}\in\mathbb{R}^L,\; L\ll D$ | compresses input to a compact code |
| **Decoder** $g_\phi$ | $\mathbf{z}\in\mathbb{R}^L \to \hat{\mathbf{x}}\in\mathbb{R}^D$ | reconstructs the input from the code |

Both are trained jointly to minimise the **reconstruction loss** over the training set:

$$\mathcal{L}(\theta,\phi) = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - g_\phi(f_\theta(\mathbf{x}_i))\|^2$$

Because the only path from input to output passes through the $L$-dimensional bottleneck,
the encoder is forced to retain only the most important structure of the data — and the
decoder must learn to recover the full input from that compressed representation.

> __Connection to CBOW and Skip-Gram__
>
> You have already built models with this compress-then-reconstruct structure. In CBOW,
> the input weight matrix $\mathbf{W}_1$ acts as an encoder that maps a sparse one-hot
> word vector into a dense $d_h$-dimensional embedding, and $\mathbf{W}_2$ decodes that
> embedding back into a prediction over the vocabulary. An autoencoder is the same idea
> applied to continuous inputs (images) with deeper, non-linear encoder and decoder networks.

___
## Task 1: Load and Explore the Training Data
We load $K = 100$ examples of MNIST digit **3** from the training split. Each 28×28
greyscale image is flattened to a 784-dimensional vector and stored as a column of the
data matrix $\mathbf{X}\in\mathbb{R}^{784\times K}$, with pixel values in $[0, 1]$.

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
X = load_mnist_digit(DIGIT; n_examples=K);
println("Data matrix size: ", size(X));   # (784, 100)

> __What is going on in this code block?__
>
> `load_mnist_digit(DIGIT; n_examples=K)` (defined in `src/Compute.jl`) queries
> `MLDatasets.MNIST(:train)` for all images of the target digit, selects the first $K$,
> flattens each 28×28 array to a 784-element column vector, and returns the `(784 × K)`
> Float32 matrix `X`. Pixel values are already normalised to $[0,1]$ by MLDatasets.

In [ ]:
let
    μ_data = mean(X);
    σ_data = std(X);
    @printf("Mean pixel value: %.4f\n", μ_data);
    @printf("Std  pixel value: %.4f\n", σ_data);
end

In [ ]:
show_image_grid(X; nrows=4, ncols=4)

### Things to think about
* __Question:__ The data matrix `X` has shape `(784, 100)`. Why do we store images as columns rather than rows? How does this column-major convention interact with Flux.jl's convention for batched layer inputs?
* __Question:__ The mean pixel value is well below 0.5. What does this tell you about the sparsity of handwritten digit images, and how might it affect the typical magnitude of the reconstruction loss before training begins?

___
## Task 2: Implement the Autoencoder Architecture
The autoencoder forward pass requires two operations. The **encoder** compresses each
input $\mathbf{x}\in\mathbb{R}^{784}$ to a bottleneck code
$\mathbf{z}\in\mathbb{R}^{L}$ by passing it through three `Dense` layers. The **decoder**
inverts this path and reconstructs the input from $\mathbf{z}$ alone. Both functions are
one-liners — the architecture is already stored in the `MyAEModel` struct.

In [ ]:
ae = build_ae_model(D, H, L);
println("AE created.")
println("  Encoder : ", ae.encoder)
println("  Decoder : ", ae.decoder)

> __What is going on in this code block?__
>
> `build_ae_model(D, H, L)` (in `src/Compute.jl`) constructs a `MyAEModel` with a
> symmetric architecture. The encoder compresses $784\to256\to128\to L=8$ using ReLU
> activations, with **no activation on the bottleneck** so that codes are unconstrained
> real numbers. The decoder inverts this path ($8\to128\to256\to784$) and ends with a
> **sigmoid** that constrains reconstructions to $[0,1]$, matching the normalised pixel range.

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
function encode(model::MyAEModel, x::AbstractMatrix)
    return model.encoder(x)   # D×N  →  L×N
end

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
function decode(model::MyAEModel, z::AbstractMatrix)
    return model.decoder(z)   # L×N  →  D×N  (sigmoid in [0,1])
end

In [ ]:
let
    x_test = X[:, 1:5];
    z_test = encode(ae, x_test);
    x̂_test = decode(ae, z_test);

    @test size(z_test)  == (L, 5);
    @test size(x̂_test) == (D, 5);
    @test all(0f0 .≤ x̂_test .≤ 1f0);

    println("✓  All Task 2 tests passed!");
end

### Things to think about
* __Question:__ The bottleneck layer has **no activation function**, while the hidden layers use ReLU and the output layer uses sigmoid. Why is it important to leave the bottleneck unconstrained? What would happen to the learned codes if you used `tanh` or `sigmoid` at the bottleneck?
* __Question:__ Compare the structure of `encode` here to the role of $\mathbf{W}_1$ in CBOW. Both map a high-dimensional input to a low-dimensional representation. What is the key architectural difference between a CBOW encoder (single weight matrix) and the AE encoder (three-layer `Chain`)?

___
## Task 3: Reconstruction Loss and Training

### Implementing `reconstruction_loss`
The loss function measures how well the autoencoder reconstructs its inputs. We use
**mean squared error (MSE)**: for each example, sum the squared pixel differences, then
average over the batch:

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2
= \texttt{mean}(\texttt{sum}((\mathbf{X} - \hat{\mathbf{X}})^{\odot 2};\,\text{dims}=1))$$

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
function reconstruction_loss(model::MyAEModel, x::AbstractMatrix)
    z  = encode(model, x);                         # D×N → L×N
    x̂  = decode(model, z);                         # L×N → D×N
    return mean(sum((x .- x̂).^2; dims=1));         # scalar MSE
end

> __What is going on in this code block?__
>
> The function runs the full AE forward pass — encode then decode — and computes the MSE.
> `sum(...; dims=1)` sums the $D=784$ squared pixel errors for each of the $N$ examples
> independently (producing a $1\times N$ row vector), and `mean(...)` then averages across
> examples so that the loss scale is independent of batch size. Because `reconstruction_loss`
> is called inside `Flux.withgradient`, Flux traces through the encode–decode chain and
> computes gradients with respect to all parameters in `model.encoder` and `model.decoder`.

In [ ]:
let
    x_test = X[:, 1:5];
    loss   = reconstruction_loss(ae, x_test);

    @test loss isa AbstractFloat;
    @test loss > 0;
    @test loss < D;

    @printf("  Reconstruction loss (untrained): %.4f\n", loss);
    println("✓  reconstruction_loss tests passed!");
end

### Training
We train for `NUM_EPOCHS = 2000` epochs using the Adam optimiser. The entire dataset
($K=100$ examples) is used as a single batch — this is feasible because the dataset is small.

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
opt_state = Flux.setup(Adam(LR), ae);
losses    = Float32[];

println("Training autoencoder...");
for epoch in 1:NUM_EPOCHS
    loss, grads = Flux.withgradient(ae) do m
        reconstruction_loss(m, X)
    end;
    Flux.update!(opt_state, ae, grads[1]);
    push!(losses, loss);
    epoch % 400 == 0 && @printf("  Epoch %4d | loss = %.4f\n", epoch, loss);
end
println("Training complete.");

In [ ]:
plot(losses;
    xlabel="Epoch", ylabel="Reconstruction Loss (MSE)",
    title="Autoencoder Training", label="MSE",
    color=:steelblue, lw=2, framestyle=:box)

### Things to think about
* __Question:__ The training loss curve should decrease and flatten. Why does it not reach zero, even after 2000 epochs? Is this a sign that training failed, or is it expected?
* __Question:__ We used the entire dataset ($K=100$) as a single batch on every epoch. For a larger dataset this would be impractical — you would use mini-batches. How would switching to mini-batches change the gradient estimate, and why might that sometimes help training?

___
## Task 4: Latent Space Analysis
After training, we evaluate the autoencoder qualitatively. First we compare original images
to their reconstructions. Then we test whether the bottleneck is **smooth** by linearly
interpolating between two training images in latent space — if the path between their codes
passes through recognisable intermediate images, the encoder has learned a meaningful
geometry.

In [ ]:
# Reconstruction: original vs AE output
let
    n_show  = 8;
    x_orig  = X[:, 1:n_show];
    z_orig  = encode(ae, x_orig);
    x_recon = decode(ae, z_orig);

    p_orig  = show_image_grid(x_orig;  nrows=2, ncols=4);
    p_recon = show_image_grid(x_recon; nrows=2, ncols=4);
    plot(p_orig, p_recon; layout=(1, 2), size=(700, 250),
         plot_title="Left: originals   Right: reconstructions")
end

> __What is going on in this code block?__
>
> We encode eight training images to their $L=8$-dimensional bottleneck codes, then decode
> back to pixel space. The reconstruction quality shows how much information the 8-dimensional
> bottleneck retains from the original 784-dimensional input. Well-trained reconstructions
> should be recognisable as digit 3 and capture the overall stroke structure, though some
> fine-grained detail is expected to be blurred by the compression.

In [ ]:
# Latent-space interpolation
# ── SOLUTION ─────────────────────────────────────────────────────────────────
let
    n_steps = 10;
    x1 = X[:, 1:1];   # first training image  (784 × 1)
    x2 = X[:, 6:6];   # sixth training image  (784 × 1)

    z1 = encode(ae, x1);   # L × 1
    z2 = encode(ae, x2);   # L × 1

    alphas = range(0f0, 1f0; length=n_steps);
    z_path = hcat([(1f0 - α) .* z1 .+ α .* z2 for α in alphas]...);  # L × n_steps
    x_path = decode(ae, z_path);                                       # D × n_steps

    show_image_grid(x_path; nrows=2, ncols=5)
end

> __What is going on in this code block?__
>
> We encode two training images to their bottleneck codes $\mathbf{z}_1$ and $\mathbf{z}_2$,
> then construct 10 intermediate codes by linear interpolation:
> $\mathbf{z}_\alpha = (1-\alpha)\mathbf{z}_1 + \alpha\mathbf{z}_2$ for
> $\alpha\in\{0, 0.11, 0.22,\ldots, 1\}$.
> The 10 codes are decoded in a single batched call and displayed as a 2×5 grid. If the
> autoencoder has learned a smooth latent space, the frames should transition gradually
> from the first digit to the sixth; if the space is fragmented, intermediate frames may
> look like blurry or incoherent mixtures.

### Things to think about
* __Question:__ Look at the interpolation sequence. Does the image transition smoothly from the first digit to the sixth, or do intermediate frames look unrecognisable? What does this tell you about the geometry of the learned latent space?
* __Question:__ A standard AE has no explicit constraint on the shape of the latent space. If you tried to generate a *new* digit by sampling $\mathbf{z}\sim\mathcal{N}(\mathbf{0},\mathbf{I})$ and decoding, would you expect realistic output? _(Hint: look at what range the actual bottleneck codes $\mathbf{z}$ take — run `extrema(encode(ae, X))` to check.)_
* __Question:__ The AE we built is the direct predecessor of the Variational Autoencoder (VAE) studied in the paper from our lab. The VAE adds a probabilistic constraint on the bottleneck. Based on what you observed in this problem set, what problem does that constraint solve?

___
## Summary
In this problem set we implemented and trained a deterministic Autoencoder on MNIST digit-3 images, establishing the encoder–decoder pattern that underlies deeper generative models.

> __Key Takeaways__
>
> * **The autoencoder is CBOW for images:** Both models learn a low-dimensional embedding by training a compress-then-reconstruct pipeline. CBOW compresses one-hot word vectors through a single linear layer; the AE compresses pixel vectors through a deep non-linear encoder. The bottleneck forces both models to capture the most salient structure of the input.
> * **MSE reconstruction loss drives encoder and decoder jointly:** Because `encode` and `decode` are composed inside a single `Flux.withgradient` call, the autoencoder gradient flows from the output error all the way back through both networks in one pass — exactly the backpropagation algorithm you saw in the feed-forward network lecture.
> * **The latent space has no guaranteed structure:** Without explicit regularisation, the bottleneck codes can take any values that minimise reconstruction loss. Linear interpolation in latent space may produce realistic intermediate images if the encoder happens to organise similar digits nearby — but this is not guaranteed. The Variational Autoencoder (VAE) addresses this by adding a KL penalty that forces the latent codes to follow a standard normal distribution, making the space smooth and suitable for generation.